# Document Scanner — Colab GPU Loss Ablation Launcher

Runs the four-loss ablation (ADR-006, `[REQ-45]`) on a Colab T4:

| Run | Loss | Config |
|---|---|---|
| exp-005 | L-A, MSE | `configs/exp/exp-005_enh_mse.yaml` |
| exp-006 | L-B, L1 | `configs/exp/exp-006_enh_l1.yaml` |
| exp-007 | L-C, L1 + MS-SSIM (alpha=0.84) | `configs/exp/exp-007_enh_l1msssim.yaml` |
| exp-008 | L-D, + Sobel (lambda=0.1) | `configs/exp/exp-008_enh_l1msssim_sobel.yaml` |

**Step 5 trains all four arms in one process, over one shared data stream.** The four arms
differ only in the loss, so running `train.py` four times would regenerate the same 80,000
synthetic samples four times over — and on Colab's two vCPUs the generator, not the T4, is
what sets the pace. Sharing the stream removes that redundancy and makes the run GPU-bound.
It also makes the comparison *paired*: every arm sees the same batches in the same order from
identical initial weights, which is the strongest form of ADR-006's "one variable at a time".

**Budget.** 2000 samples/epoch at batch 8 is 250 optimiser steps per epoch, over 40 epochs.
Estimated **2.5-3.5 h for the whole suite** shared, against **5-6 h** run one arm at a time.
These are estimates from measured generator throughput and T4 step times, not measurements —
read the first epoch's `epoch_seconds` and extrapolate before committing to the whole run.

A free session will not always survive that. Every epoch checkpoints, and Step 4 keeps
`runs/` on fast local disk while syncing to Drive every 5 epochs, so a dropped session costs
at most five epochs of recompute.

> Do not change `batch_size` between arms. It changes BatchNorm statistics and would confound
> the comparison. It lives in `configs/env/colab_t4.yaml` so all four share it by construction,
> and `train_ablation.py` refuses to start if the configs disagree on anything but the loss.

### Step 0: Confirm the GPU is actually visible

The trainers now **raise** if a profile asks for CUDA and CUDA is not usable, instead of
quietly falling back to CPU. That silent fallback is what made the previous ablation worthless.

In [ ]:
import os, torch
print('PyTorch:', torch.__version__)
print('vCPUs:', os.cpu_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    raise SystemExit('No CUDA device. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.')

> If `vCPUs` above is 4 or more, raise `num_workers` in `configs/env/colab_t4.yaml` to match.
> The generator is CPU-bound and each worker is pinned to one thread, so workers scale with
> cores almost linearly until the GPU becomes the limit.

### Step 1: Mount Drive, clone the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/DocEn'):
    !git clone https://github.com/HedieTahmouresi/DocEn.git /content/DocEn
%cd /content/DocEn
!git pull
!git log --oneline -1

### Step 2: Data and dependencies

`data.zip` must contain `clean_scans/`, `backgrounds/`, `real_photos/`, `splits/` and — if you
already generated them — `frozen/`. If `frozen/` is absent the cell rebuilds it.

> The frozen val/test sets are the comparability contract (ADR-003). Regenerate them **only**
> when the generator changes, and never in the middle of an ablation: every earlier run becomes
> incomparable. If you do regenerate, bump `frozen_version` in the experiment configs.

In [ ]:
!pip install -q -r requirements.txt

import os

if not os.path.exists('data/clean_scans'):
    for src in ('/content/drive/MyDrive/data.zip', '/content/data.zip'):
        if os.path.exists(src):
            print('Extracting', src)
            !unzip -q "{src}" -d .
            break
    else:
        raise SystemExit('data.zip not found in Drive or /content.')

if not os.path.exists('data/frozen/val'):
    print('Frozen evaluation sets missing - generating them now (a few minutes).')
    !python -m src.data.freeze

!ls data && ls data/frozen

### Step 3: Sanity ladder — run this before any long run

training-spec §9 and the phase-04 gate. The critical check is overfit-one-batch: if the model
cannot fit a single batch, the bug is in the model, the loss or the data, and no amount of GPU
time will fix it. A 40-epoch run that was never going to work is the most expensive mistake
available on a free-tier GPU.

In [ ]:
!python -m pytest tests/ -q -x --durations=10

### Step 3b: Smoke run — two short epochs, all four arms

Confirms the loop, the loader, the checkpointing and the CSV logging work on *this* runtime,
and gives a first real `epoch_seconds` to extrapolate from, before hours are committed. The
smoke run directories are deleted afterwards so they cannot be mistaken for the real run.

In [ ]:
!python train_ablation.py --env colab_t4 --epochs 2 --samples-per-epoch 200
!head -5 runs/exp-005_enh_mse/metrics.csv
!rm -rf runs/exp-00*

### Step 4: Checkpoint durability

Colab's local disk vanishes with the session (ADR-001, training-spec §7), but writing every
checkpoint straight to Drive is slow enough to cost a meaningful share of each epoch — four
arms' `last.pt` is ~700 MB, and Drive writes at a fraction of local disk speed.

So: `runs/` stays on local disk and `--mirror-dir` syncs it to Drive every 5 epochs. A dropped
session costs at most five epochs of recompute. Nothing to run here — just check the target
exists and see what is already there from an earlier session.

In [ ]:
import os, shutil

DRIVE_RUNS = '/content/drive/MyDrive/DocEn_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('Drive mirror:', DRIVE_RUNS)
print('already there:', sorted(os.listdir(DRIVE_RUNS)))

# Resuming a suite that was interrupted in an earlier session? Pull the runs back first.
RESUMING = False
if RESUMING:
    for name in os.listdir(DRIVE_RUNS):
        shutil.copytree(os.path.join(DRIVE_RUNS, name), os.path.join('runs', name), dirs_exist_ok=True)
    print('restored:', sorted(os.listdir('runs')))

### Step 5: The ablation — all four arms, one data stream

If the session dies, re-run Steps 1, 2 and 4 (with `RESUMING = True`), then add `--resume` to
the cell below. It continues every arm from its own `last.pt`.

In [ ]:
!python train_ablation.py --env colab_t4 \
    --mirror-dir /content/drive/MyDrive/DocEn_runs --mirror-every 5

#### Fallback: one arm at a time

Only if the shared-stream run hits trouble — it is slower overall, and the arms are then only
comparable up to RNG luck in the data stream rather than paired batch for batch.

In [ ]:
MIRROR = '--mirror-dir /content/drive/MyDrive/DocEn_runs --mirror-every 5'
!python train.py --config configs/exp/exp-005_enh_mse.yaml            --env colab_t4 {MIRROR}
!python train.py --config configs/exp/exp-006_enh_l1.yaml             --env colab_t4 {MIRROR}
!python train.py --config configs/exp/exp-007_enh_l1msssim.yaml       --env colab_t4 {MIRROR}
!python train.py --config configs/exp/exp-008_enh_l1msssim_sobel.yaml --env colab_t4 {MIRROR}

### Step 6: Figures and the summary table

Produces the `[REQ-22]` loss curves, the `[REQ-45]` zoomed comparison, per-sample restorations,
and `p04_ablation_summary.json` — PSNR/SSIM per variant against the `[REQ-26]` no-model
baseline. **Validation only**: the synthetic test split stays untouched until Phase 05
(`[CON-07]`), and the ablation winner is selected on validation.

In [ ]:
!python -m scripts.evaluate_ablation
!python -m scripts.save_restored_samples

### Step 7: Package the results

Checkpoints are already mirrored to Drive. This packages the small artefacts — metrics, the
resolved configs and the figures — for committing back to the repository.

In [ ]:
!zip -r phase04_results.zip outputs/figures/ runs/*/metrics.csv runs/*/metrics.json runs/*/config.yaml
!cp phase04_results.zip /content/drive/MyDrive/

from google.colab import files
files.download('phase04_results.zip')